# 🧪 W3-D2 概念实验：SFT 数据质量 vs 数量 & LoRA 参数效率

> 配套阅读：`ima/第3周-Day2-SFT监督微调的关键细节.md`
>
> SFT 核心教训：**质量远比数量重要**。量化这个权衡，可视化 LoRA 数学原理。

## 实验 1：数据质量 vs 数量

效果 ≈ 质量 × log(数量)。质量 0.7 × 500 条 > 质量 0.1 × 30000 条。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
quantities = np.logspace(2, 4.5, 200)
qualities = [0.1, 0.3, 0.5, 0.7, 0.9]
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db']
fig, ax = plt.subplots(figsize=(10, 5.5))
for q, c in zip(qualities, colors):
    effect = q * np.log10(quantities + 1) * 20
    ax.plot(quantities, effect, '-', color=c, lw=2, label=f'质量={q:.1f}')
ax.axhline(y=70, color='gray', ls='--', alpha=0.5, label='「够用」线')
ax.set_xscale('log'); ax.set_xlabel('数据量（条）'); ax.set_ylabel('模型效果分数')
ax.set_title('SFT：高质量 1000 条 ≈ 低质量 30000 条')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("红色线（质量0.1）给30000条也追不上蓝色线（质量0.7）的500条。")

## 实验 2：SFT 前后能力雷达图

SFT 改「行为」不改「知识」。指令理解和格式遵循大幅提升。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
cats = ['指令理解','格式遵循','安全性','对话连贯','创造力','知识储备']
pre, post = [20,12,10,15,35,72], [88,90,80,82,68,73]
angles = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist()
angles_c = angles + angles[:1]
fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
ax.plot(angles_c, pre+pre[:1], 'r-o', lw=2, label='预训练模型')
ax.fill(angles_c, pre+pre[:1], alpha=0.12, color='red')
ax.plot(angles_c, post+post[:1], 'b-s', lw=2, label='SFT 后')
ax.fill(angles_c, post+post[:1], alpha=0.12, color='blue')
ax.set_xticks(angles); ax.set_xticklabels(cats, fontsize=11)
ax.set_title('SFT 前后：行为大幅提升，知识基本不变', fontsize=13, pad=18)
ax.legend(loc='lower right', fontsize=11); plt.tight_layout(); plt.show()

## 实验 3：LoRA 低秩分解 — 只训练 0.3% 就够

$\Delta W = B \times A$，秩 r 远小于 d。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
d = 4096; ranks = [4,8,16,32,64,128]
total = d*d; trainable = [2*d*r for r in ranks]
pct = [t/total*100 for t in trainable]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(len(ranks)), pct, color='#3498db', alpha=0.8)
ax.set_xticks(range(len(ranks))); ax.set_xticklabels([f'r={r}' for r in ranks])
ax.set_ylabel('可训练参数占比 (%)')
ax.set_title(f'LoRA 参数效率（d={d}）：r=8 时只训练 {pct[1]:.2f}%')
for bar, p in zip(bars, pct):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{p:.3f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_yscale('log'); ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()
print(f"全参数: {total/1e6:.1f}M | LoRA r=8: {trainable[1]/1e6:.1f}M ({pct[1]:.3f}%)")

## 实验 4：过拟合检测

训练集降，验证集升 → 过拟合信号。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(1); epochs = np.arange(1, 31)
train_loss = 2.8*np.exp(-epochs/5)+0.25+0.005*epochs*0.01
val_loss = 2.6*np.exp(-epochs/5)+0.35+0.0008*np.maximum(0,epochs-8)**2+np.random.normal(0,0.015,len(epochs))
val_loss = np.maximum(val_loss, 0.2); best = int(np.argmin(val_loss))+1
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, train_loss, 'b-o', lw=2, ms=4, label='训练集 Loss')
ax.plot(epochs, val_loss, 'r-s', lw=2, ms=4, label='验证集 Loss')
ax.axvline(best, color='orange', ls='--', lw=2, label=f'最佳停止点 epoch={best}')
ax.axvspan(best, 30, alpha=0.08, color='red', label='过拟合区域')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('SFT 过拟合：验证集回升 → 立刻停止')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f"最佳 epoch={best}。继续训练验证集更差——模型在「背答案」。")

## 结论

| 问题 | 实验证据 |
|---|---|
| 质量 vs 数量 | 实验1：质量0.7×500 > 质量0.1×30000 |
| SFT 改什么 | 实验2：行为大幅提升，知识不变 |
| LoRA 效率 | 实验3：r=8 只需 0.39% 参数 |
| 过拟合判断 | 实验4：验证集 Loss 回升即停 |

→ 配套阅读：`ima/第3周-Day2-SFT监督微调的关键细节.md`